In [ ]:
# Instalação de Dependências e Importações
%pip install -q psycopg2-binary boto3

import io
import csv
import psycopg2
import boto3

# Configuracoes de Infraestrutura (Rede interna do Docker)
config_db = {"host": "postgres", "database": "postgres", "user": "postgres", "password": "senha123", "port": 5432}
config_s3 = {"endpoint_url": "http://minio:9000", "aws_access_key_id": "root-minio", "aws_secret_access_key": "root12345678", "region_name": "local"}

bucket_name = "raw"
destino_s3 = "python_native/transacoes_csv/transacoes_ledger.csv"

In [10]:
# Extração (Extract) do PostgreSQL via Cursor Nativo
print("Conectando ao PostgreSQL e extraindo dados financeiros...")

with psycopg2.connect(**config_db) as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT id_transacao, conta_origem, conta_destino, valor, moeda, data_transacao FROM transacoes_financeiras;")
        colunas = [desc[0] for desc in cursor.description]
        dados = cursor.fetchall()

print(f"Quantidade de registros extraidos: {len(dados)}")

Conectando ao PostgreSQL e extraindo dados financeiros...
Quantidade de registros extraidos: 20000


In [11]:
# Carga (Load) para o MinIO via Buffer de Memória
print("Iniciando construcao do CSV em buffer de memoria RAM (StringIO)...")
buffer_memoria = io.StringIO()
escritor = csv.writer(buffer_memoria, delimiter=';')

# Grava cabecalho e linhas no buffer
escritor.writerow(colunas)
escritor.writerows(dados)

print(f"Transmitindo arquivo via HTTP para s3://{bucket_name}/{destino_s3}...")
s3_client = boto3.client('s3', **config_s3)
s3_client.put_object(
    Bucket=bucket_name,
    Key=destino_s3,
    Body=buffer_memoria.getvalue(),
    ContentType='text/csv; charset=utf-8'
)

print("Ingestao CSV concluida com sucesso!")

Iniciando construcao do CSV em buffer de memoria RAM (StringIO)...
Transmitindo arquivo via HTTP para s3://raw/python_native/transacoes_csv/transacoes_ledger.csv...
Ingestao CSV concluida com sucesso!


In [12]:
# Encerramento das Conexoes
print("Fechando a conexao com o banco de dados PostgreSQL...")
conn.close()
print("Conexao encerrada com seguranca. Os recursos do banco foram liberados.")

Fechando a conexao com o banco de dados PostgreSQL...
Conexao encerrada com seguranca. Os recursos do banco foram liberados.


### Analise de Arquitetura: O Formato CSV no Python Nativo

**Caracteristicas Principais:** Formato baseado em texto plano, estritamente orientado a linhas.

#### Vantagens
* **Universalidade:** Qualquer ferramenta de mercado ou planilha consome sem barreiras tecnicas.
* **Simplicidade:** Processamento leve, sem a necessidade de engines ou frameworks robustos.

#### Desvantagens
* **Perda de Tipagem:** O banco de dados entrega tipos complexos, mas o CSV armazena tudo como texto bruto, demandando inferencia ou remapeamento posterior.
* **Ineficiencia Espacial:** Ocupa muito espaco no Data Lake por nao possuir compressao binaria nativa.

#### Caso de Uso no Mercado
* Exportacao de dados consolidados para consumo imediato de analistas de negocios via Excel ou ferramentas de BI.
* Cargas de tabelas estaticas de baixissimo volume, como tabelas de feriados ou dicionarios de codigos.